# EuroSAT Vegetation Classification — Exploration and Error Analysis

**Project:** Sentinel-2 land-cover classification with PyTorch CNNs  
**Dataset:** [EuroSAT RGB](https://github.com/phelber/EuroSAT) (Helber et al. 2019)  
**Repository:** `sentinel-vegetation-cnn`

---

## Scientific question

> *How well can a convolutional neural network distinguish vegetation and land-cover classes from Sentinel-2 imagery, and what can its errors tell us about the spectral and spatial similarities among those classes?*

This notebook documents the full experimental workflow:
1. **Dataset exploration** — class composition, sample patches, spectral profiles  
2. **Baseline evaluation** — logistic regression and random forest on mean RGB values  
3. **CNN evaluation** — test-set metrics, confusion matrix, per-class performance  
4. **Error analysis** — which classes are confused, and why from an ecological/remote-sensing perspective  
5. **Limitations** — validation strategy, spatial autocorrelation, generalisability  

---

**Pre-requisites:** Run training and evaluation first:
```bash
python -m src.train
python -m src.evaluate --run_baseline
```
The notebook will detect whether results files are present and raise a clear error if they are not.  
To run training inline, set `RUN_TRAINING_INLINE = True` in the Setup cell.

## 1  Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path so we can import from src/
# __file__ is not defined in Jupyter; locate the root by searching upward
# from the current working directory for the directory that contains src/.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "src").exists() else _cwd.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(
        f"Cannot find project root from {_cwd}. "
        "Launch Jupyter from the project root: `jupyter notebook` (not from notebooks/)."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Configuration
DATA_DIR       = PROJECT_ROOT / "data"
RESULTS_DIR    = PROJECT_ROOT / "results"
FIGURES_DIR    = PROJECT_ROOT / "figures"
MODELS_DIR     = PROJECT_ROOT / "models"
CHECKPOINT     = MODELS_DIR / "small_cnn_best.pt"

# Set to True to run training inside the notebook (slow but self-contained)
RUN_TRAINING_INLINE = False

# Standard imports
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import pandas as pd
import torch
from torchvision import datasets
from torch.utils.data import Subset, DataLoader
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score,
)

# Project modules
from src.data import (
    CLASS_NAMES, CLASS_DISPLAY_NAMES, VEGETATION_CLASSES,
    EUROSAT_MEAN, EUROSAT_STD,
    load_eurosat, make_splits, get_data_loaders,
    get_raw_transform, get_transforms,
    load_split_indices, extract_mean_features,
)
from src.model import build_model, SmallCNN

# Plot style
%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Optionally run training inline
if RUN_TRAINING_INLINE:
    import argparse
    from src.train import train
    args = argparse.Namespace(
        data_dir=str(DATA_DIR), results_dir=str(RESULTS_DIR),
        models_dir=str(MODELS_DIR), architecture="small_cnn",
        epochs=40, batch_size=64, lr=1e-3, weight_decay=1e-4,
        dropout=0.4, image_size=64, num_workers=0,
        train_frac=0.70, val_frac=0.15, seed=42, early_stopping=10,
    )
    history = train(args)

    from src.evaluate import evaluate_model
    eval_args = argparse.Namespace(
        checkpoint=str(CHECKPOINT), data_dir=str(DATA_DIR),
        results_dir=str(RESULTS_DIR), figures_dir=str(FIGURES_DIR),
        run_baseline=True,
    )
    evaluate_model(eval_args)

## 2  EuroSAT dataset overview

### Sentinel-2 and EuroSAT

Sentinel-2 is part of the European Space Agency's Copernicus programme. It carries a MultiSpectral Instrument (MSI) that captures imagery in 13 spectral bands spanning the visible, near-infrared, and short-wave infrared at 10–60 m spatial resolution. The 10 m bands (B2 Blue, B3 Green, B4 Red, B8 NIR) are the most commonly used for vegetation analysis.

**EuroSAT** (Helber et al. 2019) is a benchmark land-use / land-cover dataset derived from Sentinel-2. It contains 27,000 geo-referenced 64×64-pixel image patches drawn from 34 European countries, labelled into 10 classes. Each patch covers approximately 0.64 km² at 10 m resolution.

The RGB version used here discards NIR and SWIR bands. The full 13-band multispectral version is available separately and is explored in the stretch-goal analysis (see README).

### Why spectral information matters for vegetation classification

Plants have characteristic spectral signatures driven by leaf biochemistry and canopy structure:
- **Green reflectance peak (~550 nm):** chlorophyll absorbs red and blue but reflects green, giving vegetation its colour.
- **Red-edge and NIR plateau (~700–900 nm):** healthy vegetation strongly reflects NIR due to cell structure scattering. The abrupt increase from red absorption to NIR reflectance — the red-edge — is one of the most diagnostic vegetation signals and is captured by Sentinel-2 bands B5/B6/B7 (not in the RGB subset).
- **Canopy structure:** Forest has a more complex, multi-layered canopy than grassland, producing different bi-directional reflectance patterns, shadow fractions, and texture.

These signals are why even simple spectral classifiers can partially distinguish vegetation types — but spatial texture (captured by convolution) adds further discriminating power, especially for classes like Forest vs. Herbaceous Vegetation where structural complexity differs substantially.

### EuroSAT classes

In [ ]:
class_info = {
    "AnnualCrop":           {"display": "Annual Crop",           "veg": True,  "desc": "Arable fields with seasonal crops (e.g. wheat, maize, sunflower)"},
    "Forest":               {"display": "Forest",                "veg": True,  "desc": "Continuous tree cover; mixed or broadleaf/conifer"},
    "HerbaceousVegetation": {"display": "Herbaceous Vegetation", "veg": True,  "desc": "Non-woody vegetation: meadows, rough grassland, shrubland"},
    "Highway":              {"display": "Highway",               "veg": False, "desc": "Major roads and surrounding built environment"},
    "Industrial":           {"display": "Industrial",            "veg": False, "desc": "Industrial buildings, warehouses, car parks"},
    "Pasture":              {"display": "Pasture",               "veg": True,  "desc": "Managed grassland for livestock grazing"},
    "PermanentCrop":        {"display": "Permanent Crop",        "veg": True,  "desc": "Orchards, vineyards, olive groves — crops not re-sown annually"},
    "Residential":          {"display": "Residential",          "veg": False, "desc": "Urban housing areas; often includes trees/gardens"},
    "River":                {"display": "River",                "veg": False, "desc": "Flowing water bodies with riparian borders"},
    "SeaLake":              {"display": "Sea / Lake",           "veg": False, "desc": "Open water: lakes, reservoirs, coastal sea"},
}

df = pd.DataFrame(class_info).T.reset_index().rename(columns={"index": "Class"})
df["Vegetation?"] = df["veg"].map({True: "Yes", False: "No"})
df = df[["display", "Vegetation?", "desc"]].rename(columns={"display": "Class", "desc": "Description"})
df.index = range(1, len(df) + 1)
df.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

## 3  Class distribution

In [ ]:
# Load dataset (download if not already present)
full_dataset = load_eurosat(DATA_DIR, transform=None, download=True)
print(f"Dataset size: {len(full_dataset):,} images  |  Classes: {full_dataset.classes}")

# Count samples per class
targets = np.array(full_dataset.targets)
class_names = full_dataset.classes
class_counts = {cn: (targets == i).sum() for i, cn in enumerate(class_names)}

fig, ax = plt.subplots(figsize=(11, 4))
names   = [CLASS_DISPLAY_NAMES[c] for c in class_names]
counts  = [class_counts[c] for c in class_names]
colors  = ["#2E7D32" if c in VEGETATION_CLASSES else "#78909C" for c in class_names]

bars = ax.bar(names, counts, color=colors, edgecolor="white", width=0.7)
ax.set_ylabel("Number of image patches")
ax.set_title("EuroSAT: Class Distribution", fontweight="bold")
ax.set_xticklabels(names, rotation=35, ha="right")
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{count:,}", ha="center", va="bottom", fontsize=8.5)

veg_patch   = mpatches.Patch(color="#2E7D32", label="Vegetation class")
other_patch = mpatches.Patch(color="#78909C", label="Non-vegetation class")
ax.legend(handles=[veg_patch, other_patch], loc="upper right")
fig.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"EuroSAT is approximately balanced: {min(counts):,}–{max(counts):,} images per class.")

EuroSAT is approximately class-balanced, which simplifies training and means that overall accuracy is a reasonable summary metric. Vegetation classes account for five of the ten categories.

## 4  Sample image patches

Before building any model, it is worth visually inspecting the data. The patches below are representative examples of each class — the same format the CNN will receive.

In [ ]:
raw_dataset = load_eurosat(DATA_DIR, transform=get_raw_transform(64), download=False)
n_classes   = len(class_names)
n_examples  = 5

# Draw a fixed random sample per class for reproducibility
rng = np.random.default_rng(0)

fig, axes = plt.subplots(n_classes, n_examples, figsize=(n_examples * 1.8, n_classes * 1.8))

for row, (cls_idx, cls_name) in enumerate(enumerate(class_names)):
    class_indices = np.where(targets == cls_idx)[0]
    chosen = rng.choice(class_indices, size=n_examples, replace=False)

    for col, img_idx in enumerate(chosen):
        img_tensor, _ = raw_dataset[img_idx]
        img = img_tensor.permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        ax  = axes[row, col]
        ax.imshow(img)
        ax.axis("off")
        if col == 0:
            display_name = CLASS_DISPLAY_NAMES[cls_name]
            color = "#1B5E20" if cls_name in VEGETATION_CLASSES else "#37474F"
            ax.set_ylabel(display_name, fontsize=8.5, color=color,
                          rotation=0, labelpad=80, va="center")

fig.suptitle("Representative 64×64 Patch Examples per Class", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.savefig(FIGURES_DIR / "sample_patches.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations from visual inspection:**

- **Forest** has a visually distinctive dark-green, highly textured appearance driven by canopy shadow and the irregular crowns of individual trees.
- **Annual Crop** and **Permanent Crop** often show regular geometric patterns from field boundaries, but Permanent Crop tends to have a more irregular internal texture (row structure of orchards/vineyards).
- **Pasture** and **Herbaceous Vegetation** are visually very similar — both appear as relatively uniform, bright-green patches. This visual ambiguity foreshadows one of the most frequent confusion pairs in the model.
- **Sea / Lake** is spectrally very distinctive (uniform dark blue) and should be easy to classify correctly.
- **Highway** can include surrounding vegetation, making it somewhat ambiguous depending on the surrounding landscape.

## 5  Mean spectral profiles

Before modelling, it is useful to examine the mean RGB reflectance per class. This is exactly what the baseline classifiers use as input — it answers the question: *How far does spectral information alone go?*

In [ ]:
# Compute per-class mean RGB from a 2,000-image random sample (fast)
sample_size = 2000
sample_idx  = rng.choice(len(raw_dataset), size=sample_size, replace=False)
loader = DataLoader(
    Subset(raw_dataset, sample_idx.tolist()),
    batch_size=256, shuffle=False, num_workers=0,
)

sample_imgs, sample_lbls = [], []
for imgs, lbls in loader:
    sample_imgs.append(imgs)
    sample_lbls.append(lbls)
sample_imgs = torch.cat(sample_imgs)   # (N, 3, 64, 64)
sample_lbls = torch.cat(sample_lbls)   # (N,)

channel_names = ["Red", "Green", "Blue"]
channel_colors = ["#E53935", "#43A047", "#1E88E5"]

class_means = {}  # cls_name → [mean_R, mean_G, mean_B]
for cls_idx, cls_name in enumerate(class_names):
    mask    = sample_lbls == cls_idx
    patches = sample_imgs[mask]          # (n_cls, 3, 64, 64)
    means   = patches.mean(dim=[0, 2, 3]).tolist()  # per-channel spatial mean
    class_means[cls_name] = means

# ── Plot ──────────────────────────────────────────────────────────────────
x = np.arange(n_classes)
width = 0.28
display = [CLASS_DISPLAY_NAMES[c] for c in class_names]

fig, ax = plt.subplots(figsize=(13, 4.5))
for ch_i, (ch_name, ch_color) in enumerate(zip(channel_names, channel_colors)):
    vals = [class_means[c][ch_i] for c in class_names]
    ax.bar(x + (ch_i - 1) * width, vals, width,
           label=ch_name, color=ch_color, alpha=0.75, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(display, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Mean pixel value (0–1 scale)")
ax.set_title("Mean per-channel RGB reflectance by class", fontweight="bold")
ax.legend()
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
plt.savefig(FIGURES_DIR / "spectral_profiles.png", dpi=150, bbox_inches="tight")
plt.show()

**Spectral interpretation:**

- **Sea / Lake** has the lowest overall reflectance (dark) with a relatively higher blue component — a classic water spectral signature.
- **Forest** tends to have lower red reflectance (chlorophyll absorption) and a distinctive red-green ratio. However, in the RGB subset, the all-important NIR plateau is absent.
- **Annual Crop**, **Pasture**, and **Herbaceous Vegetation** have very similar RGB profiles — all appear as relatively bright green patches — which explains why the spectral baseline struggles to separate them. In the full 13-band Sentinel-2 data, the red-edge bands (B5, B6, B7) would provide much sharper discrimination.
- **Industrial** and **Highway** have high, spectrally flat reflectance (concrete / asphalt).

This similarity among vegetation classes motivates the CNN: by learning spatial texture in addition to spectral means, the network can potentially exploit structural differences that aggregate statistics cannot.

## 6  Baseline classifiers (mean spectral features)

The baseline uses only the three mean pixel values per image patch as features. This is a deliberate lower bound: it shows what can be achieved from spectral information alone, with no spatial structure.

In [ ]:
baseline_path = RESULTS_DIR / "baseline_results.json"

if baseline_path.exists():
    with open(baseline_path) as f:
        baseline_results = json.load(f)
    print("Baseline results loaded from file.")
else:
    # Compute baseline inline (requires split indices from training)
    split_path = RESULTS_DIR / "split_indices.json"
    if not split_path.exists():
        raise FileNotFoundError(
            "Run `python -m src.train` first to generate split_indices.json, "
            "then `python -m src.evaluate --run_baseline`."
        )
    train_idx, _, test_idx = load_split_indices(split_path)
    from src.evaluate import run_baseline
    baseline_results = run_baseline(DATA_DIR, train_idx, test_idx, RESULTS_DIR)

print("\nBaseline performance (test set):")
df_bl = pd.DataFrame(baseline_results).T
df_bl.index = ["Logistic Regression", "Random Forest"]
df_bl.columns = ["Accuracy", "Macro F1"]
df_bl = df_bl.map(lambda x: f"{x:.4f}")
print(df_bl.to_string())

**Interpretation:** The baseline achieves moderate accuracy — respectable given it uses only 3 numbers per image — but meaningful headroom remains, especially on vegetation classes with similar spectral profiles. The CNN will be assessed primarily against this ceiling.

## 7  CNN architecture summary

The `SmallCNN` is a compact from-scratch convolutional network designed to be transparent and trainable without a GPU in reasonable time.

In [ ]:
model_summary = SmallCNN(num_classes=10, in_channels=3)
n_params = model_summary.count_parameters()

print("SmallCNN Architecture")
print("=" * 50)
print(model_summary)
print(f"\nTotal trainable parameters: {n_params:,}")
print()
print("Spatial resolution progression:")
print("  Input:      64 × 64 × 3")
print("  Block 1:    32 × 32 × 32   (32 filters, 3×3, MaxPool 2×2)")
print("  Block 2:    16 × 16 × 64   (64 filters, 3×3, MaxPool 2×2)")
print("  Block 3:     8 ×  8 × 128  (128 filters, 3×3, MaxPool 2×2)")
print("  Block 4:     4 ×  4 × 256  (256 filters, 3×3, MaxPool 2×2)")
print("  GlobalAvgPool:       256-d")
print("  FC head:      256 → 128 → 10 classes")

## 8  Training curves

Good training behaviour — loss decreasing smoothly, train and val tracks converging — is a prerequisite for trusting the test-set results.

In [ ]:
history_path = RESULTS_DIR / "small_cnn_history.json"
if not history_path.exists():
    raise FileNotFoundError(
        f"Training history not found at {history_path}. "
        "Run `python -m src.train` first."
    )

with open(history_path) as f:
    history = json.load(f)

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

ax = axes[0]
ax.plot(epochs, history["train_loss"], label="Train", color="#1565C0", lw=1.8)
ax.plot(epochs, history["val_loss"],   label="Val",   color="#C62828", lw=1.8)
ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
ax.set_title("Loss"); ax.legend(); ax.grid(alpha=0.25)

ax = axes[1]
ax.plot(epochs, [a*100 for a in history["train_acc"]], label="Train", color="#1565C0", lw=1.8)
ax.plot(epochs, [a*100 for a in history["val_acc"]],   label="Val",   color="#C62828", lw=1.8)
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy"); ax.set_ylim(0, 100); ax.legend(); ax.grid(alpha=0.25)

ax = axes[2]
ax.semilogy(epochs, history["lr"], color="#6A1B9A", lw=1.8)
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning rate (log scale)")
ax.set_title("Cosine LR Schedule"); ax.grid(alpha=0.25)

fig.suptitle("SmallCNN Training History", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves_notebook.png", dpi=150, bbox_inches="tight")
plt.show()

best_ep  = int(np.argmax(history["val_acc"])) + 1
best_acc = max(history["val_acc"])
print(f"Best validation accuracy: {best_acc:.4f} at epoch {best_ep}")

## 9  Test-set evaluation

In [ ]:
metrics_path = RESULTS_DIR / "test_metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(
        f"Test metrics not found at {metrics_path}. "
        "Run `python -m src.evaluate` first."
    )

with open(metrics_path) as f:
    metrics = json.load(f)

acc    = metrics["test_accuracy"]
mac_f1 = metrics["macro_f1"]
cls_report = metrics["classification_report"]
class_names_loaded = metrics["class_names"]

print(f"SmallCNN — Test-set performance")
print(f"  Overall accuracy : {acc:.4f}")
print(f"  Macro F1         : {mac_f1:.4f}")

# Compare against baseline
if baseline_path.exists():
    bl = baseline_results
    print(f"\nComparison:")
    print(f"  {'Model':<30}  {'Accuracy':>10}  {'Macro F1':>10}")
    print(f"  {'-'*55}")
    print(f"  {'Logistic Regression (mean RGB)':<30}  {bl['logistic_regression']['accuracy']:>10.4f}  {bl['logistic_regression']['macro_f1']:>10.4f}")
    print(f"  {'Random Forest (mean RGB)':<30}  {bl['random_forest']['accuracy']:>10.4f}  {bl['random_forest']['macro_f1']:>10.4f}")
    print(f"  {'SmallCNN':<30}  {acc:>10.4f}  {mac_f1:>10.4f}")

In [ ]:
# Per-class precision / recall / F1 table
per_class = {
    CLASS_DISPLAY_NAMES.get(cn, cn): {
        "Precision":  round(cls_report[cn]["precision"],  4),
        "Recall":     round(cls_report[cn]["recall"],     4),
        "F1":         round(cls_report[cn]["f1-score"],   4),
        "Support":    int(cls_report[cn]["support"]),
        "Vegetation": "Yes" if cn in VEGETATION_CLASSES else "No",
    }
    for cn in class_names_loaded
}
df_cls = pd.DataFrame(per_class).T
df_cls = df_cls.sort_values("F1", ascending=False)

# Highlight vegetation rows
def highlight_veg(row):
    return ["background-color: #E8F5E9" if row["Vegetation"] == "Yes" else "" for _ in row]

df_cls.style.apply(highlight_veg, axis=1).format({
    "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}"
})

## 10  Confusion matrix

In [ ]:
# Load pre-computed confusion matrix figure, or regenerate inline
cm_fig_path = FIGURES_DIR / "confusion_matrix.png"

if cm_fig_path.exists():
    from IPython.display import Image
    Image(str(cm_fig_path))
else:
    # Regenerate inline — requires loading model and running inference
    if not CHECKPOINT.exists():
        raise FileNotFoundError(f"Checkpoint not found at {CHECKPOINT}.")
    from src.evaluate import load_model_from_checkpoint, get_predictions, plot_confusion_matrix
    model, cn, sa = load_model_from_checkpoint(CHECKPOINT, device)
    train_idx_r, _, test_idx_r = load_split_indices(RESULTS_DIR / "split_indices.json")
    y_true_r, y_pred_r, _ = get_predictions(model, DATA_DIR, test_idx_r, device)
    plot_confusion_matrix(y_true_r, y_pred_r, cn, cm_fig_path)
    from IPython.display import Image
Image(str(cm_fig_path))

## 11  Error analysis

### 11.1  Which classes are most frequently confused?

In [ ]:
# Rerun inference to get arrays for detailed analysis
if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint not found at {CHECKPOINT}. Run training first.")

from src.evaluate import load_model_from_checkpoint, get_predictions

model_eval, class_names_eval, saved_args = load_model_from_checkpoint(CHECKPOINT, device)
train_idx_e, val_idx_e, test_idx_e = load_split_indices(RESULTS_DIR / "split_indices.json")
image_size_e = saved_args.get("image_size", 64)

y_true_e, y_pred_e, y_proba_e = get_predictions(
    model_eval, DATA_DIR, test_idx_e, device, image_size_e
)

# Build confusion pair ranking
cm_raw = confusion_matrix(y_true_e, y_pred_e)
cm_off = cm_raw.copy()
np.fill_diagonal(cm_off, 0)

pairs = []
for i in range(len(class_names_eval)):
    for j in range(len(class_names_eval)):
        if i != j and cm_off[i, j] > 0:
            recall_i = cm_raw[i, i] / cm_raw[i].sum()  # class recall
            pairs.append({
                "True class":       CLASS_DISPLAY_NAMES.get(class_names_eval[i], class_names_eval[i]),
                "Predicted as":     CLASS_DISPLAY_NAMES.get(class_names_eval[j], class_names_eval[j]),
                "# Errors":         cm_off[i, j],
                "% of true class":  round(100 * cm_off[i, j] / cm_raw[i].sum(), 1),
                "Both vegetation?": (
                    "Yes" if class_names_eval[i] in VEGETATION_CLASSES
                             and class_names_eval[j] in VEGETATION_CLASSES
                    else "No"
                ),
            })

df_pairs = pd.DataFrame(pairs).sort_values("# Errors", ascending=False).head(15)
df_pairs.index = range(1, len(df_pairs) + 1)
df_pairs.style.apply(
    lambda row: ["background-color: #FFF3E0" if row["Both vegetation?"] == "Yes" else "" for _ in row],
    axis=1,
)

### 11.2  Example patches for confused pairs

In [ ]:
err_fig_path = FIGURES_DIR / "error_examples.png"

if err_fig_path.exists():
    from IPython.display import Image
    Image(str(err_fig_path))
else:
    from src.evaluate import plot_error_examples
    plot_error_examples(
        DATA_DIR, test_idx_e, y_true_e, y_pred_e,
        class_names_eval, err_fig_path, image_size=image_size_e,
    )
    from IPython.display import Image
Image(str(err_fig_path))

### 11.3  Vegetation class confusion — focused analysis

In [ ]:
veg_indices = [
    i for i, cn in enumerate(class_names_eval)
    if cn in VEGETATION_CLASSES
]

# Filter to samples whose true class is a vegetation class
veg_mask = np.isin(y_true_e, veg_indices)
y_true_veg = y_true_e[veg_mask]
y_pred_veg = y_pred_e[veg_mask]

veg_class_names = [class_names_eval[i] for i in veg_indices]
veg_display     = [CLASS_DISPLAY_NAMES.get(c, c) for c in veg_class_names]

# Remap to 0-based indices for confusion matrix
idx_remap = {orig: new for new, orig in enumerate(veg_indices)}
y_true_v = np.array([idx_remap[l] for l in y_true_veg])
y_pred_v = np.array([
    idx_remap[l] if l in idx_remap else len(veg_indices)  # "other" bucket
    for l in y_pred_veg
])

# Only show vegetation↔vegetation confusion
veg_only_mask = y_pred_v < len(veg_indices)
y_true_v = y_true_v[veg_only_mask]
y_pred_v = y_pred_v[veg_only_mask]

cm_veg = confusion_matrix(y_true_v, y_pred_v)
cm_veg_norm = cm_veg.astype(float) / cm_veg.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_veg_norm, cmap="YlOrRd", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8, label="Recall (fraction)")
ax.set_xticks(range(len(veg_display)))
ax.set_yticks(range(len(veg_display)))
ax.set_xticklabels(veg_display, rotation=35, ha="right", fontsize=9)
ax.set_yticklabels(veg_display, fontsize=9)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Vegetation-class confusion (row-normalised)", fontweight="bold")

thresh = 0.5
for i in range(cm_veg_norm.shape[0]):
    for j in range(cm_veg_norm.shape[1]):
        color = "white" if cm_veg_norm[i, j] > thresh else "black"
        ax.text(j, i, f"{cm_veg_norm[i,j]:.2f}",
                ha="center", va="center", fontsize=9, color=color)

fig.tight_layout()
plt.savefig(FIGURES_DIR / "vegetation_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

### 11.4  Ecological interpretation of errors

The confusion patterns are not random — they reflect genuine ecological and remote-sensing ambiguity.

**Pasture ↔ Herbaceous Vegetation**  
This is the most predictable confusion in any vegetation mapping exercise. Both classes are dominated by non-woody, low-growing green vegetation. In the RGB bands available here, their spectral signatures are nearly identical — both exhibit high green reflectance and moderate red absorption. The primary difference is ecological management (pasture is grazed; herbaceous vegetation may be natural or semi-natural), which does not necessarily produce a visible spectral or spatial signature at 10 m resolution. Even the full Sentinel-2 spectral suite, or time-series analysis capturing phenological differences in greenup timing, may struggle to separate these reliably at patch scale. This confusion is a known limitation in continental-scale vegetation mapping.

**Annual Crop ↔ Permanent Crop / Herbaceous Vegetation**  
At a single point in the growing season, an annual crop field may look indistinguishable from rough grassland. The key discriminating cue — field geometry and the presence of regular row structure in orchards/vineyards — is available to the CNN as spatial texture, which is why the CNN outperforms the spectral baseline on these classes. However, geometric patterns are not always present at 64×64 patch scale if the patch does not capture a field boundary or row interval.

**Forest**  
Forest is among the best-classified vegetation types. The combination of (a) distinctive spectral signal (darker, higher red absorption) and (b) spatially complex texture from canopy shadows and individual tree crowns gives the CNN strong multi-scale discriminating features. This aligns with the longstanding observation in remote sensing that forest can be reliably mapped with moderate-resolution imagery even without NIR bands.

**Residential ↔ other classes**  
Residential areas often contain substantial tree and garden cover. Patches dominated by street trees or parks can superficially resemble HerbaceousVegetation or even Forest, producing asymmetric errors in both directions.

**What spatial learning adds:**  
The improvement of SmallCNN over the mean-RGB baseline is most pronounced for classes with characteristic spatial texture (Forest, PermanentCrop, Highway, Industrial). It is smallest for spectrally distinctive classes like Sea/Lake — for those, the spectral mean alone is almost sufficient.

## 12  Stretch goal — spectral bands analysis

The following cell provides a framework for analysing whether adding additional Sentinel-2 spectral bands beyond RGB improves vegetation classification. This requires the full multispectral EuroSAT dataset (`EuroSATMS`) which must be downloaded separately from the original source (see README).

The key hypothesis is that bands B5 (red-edge 1, 705 nm), B6 (red-edge 2, 740 nm), B7 (red-edge 3, 783 nm), and B8A (narrow NIR, 865 nm) should substantially improve discrimination among vegetation types because the red-edge inflection is driven by chlorophyll content and leaf area index — quantities that differ among Forest, HerbaceousVegetation, Pasture, and Annual/Permanent Crop.

In [ ]:
# Stretch goal — MS band comparison (requires EuroSATMS dataset)
# This cell runs only if the MS dataset has been downloaded.

ms_results_path = RESULTS_DIR / "small_cnn_ms_history.json"

if ms_results_path.exists():
    with open(ms_results_path) as f:
        history_ms = json.load(f)
    with open(history_path) as f:
        history_rgb = json.load(f)

    fig, ax = plt.subplots(figsize=(10, 4))
    ep_rgb = range(1, len(history_rgb["val_acc"]) + 1)
    ep_ms  = range(1, len(history_ms["val_acc"]) + 1)
    ax.plot(ep_rgb, [a*100 for a in history_rgb["val_acc"]],
            label="RGB (3 bands)", color="#1565C0", lw=1.8)
    ax.plot(ep_ms,  [a*100 for a in history_ms["val_acc"]],
            label="MS (13 bands)", color="#2E7D32", lw=1.8, linestyle="--")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Validation accuracy (%)")
    ax.set_title("RGB vs. 13-band Multispectral — Validation Accuracy", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.savefig(FIGURES_DIR / "rgb_vs_ms.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Multispectral results not found.")
    print("To run the stretch goal:")
    print("  1. Download EuroSATMS from: https://github.com/phelber/EuroSAT")
    print("  2. Run: python -m src.train --architecture small_cnn --in_channels 13")
    print("  Requires the rasterio package and a modified src/data.py MS loader.")

## 13  Validation strategy and limitations

### Random splitting and spatial autocorrelation

The train/val/test split used here is a **random image-level split**: each 64×64 patch is independently assigned to one partition. This is the standard approach for benchmark evaluation on EuroSAT and is appropriate for comparing methods under controlled conditions.

However, for a **real landscape-scale mapping application**, random splitting almost certainly **overestimates** generalisation performance, for the following reasons:

1. **Spatial autocorrelation.** Nearby image patches share environmental gradients (soil type, climate, topography), seasonal phenological state, and atmospheric conditions at acquisition time. When patches from the same geographic neighbourhood appear in both training and test sets, the test set is not fully independent.

2. **Sensor and calibration consistency.** Patches from the same Sentinel-2 tile (acquired at the same time and processed with the same atmospheric correction parameters) are more similar to each other than to patches from different tiles. Random splitting can distribute patches from the same tile into train and test.

3. **Class distribution shift.** The vegetation composition of a novel region (say, a Mediterranean landscape vs. a northern European one) may differ substantially from the training distribution, even within EuroSAT's European coverage.

### Stronger validation strategies

For a production vegetation mapping workflow analogous to the Minnesota forest classification exercise in my earlier work, a more conservative validation strategy would include:

- **Spatially blocked cross-validation:** divide the study area into geographic blocks (e.g., grid cells or watersheds) and use leave-one-block-out evaluation. This ensures no spatial neighbours appear in both train and test.
- **Leave-region-out validation:** hold out an entire geographic region (country, ecoregion, or tile) and use it as the test set. This tests whether the model generalises to new landscapes rather than to new samples from the same landscapes it was trained on.
- **Temporal validation:** train on imagery from one year and evaluate on imagery from a different year to test robustness to inter-annual phenological variation.

The EuroSAT benchmark does not provide geographic coordinates for individual patches, so spatial blocking cannot be implemented directly. This is a known limitation of the benchmark for applied mapping tasks.

### Other limitations

- **RGB only.** The NIR and red-edge bands that carry the strongest vegetation signal are not available in the RGB variant. Classification of fine-grained vegetation types would benefit substantially from the full spectral stack.
- **Single-date imagery.** Phenological variation across the growing season is a powerful discriminator for crop classes (annual vs. permanent) but is not captured in single-date patches.
- **Patch scale vs. landscape scale.** EuroSAT patches are 64×64 pixels (0.64 km²). Real mapping at landscape scale involves edge effects, mixed pixels, and spatial context beyond a single patch that the model cannot exploit.
- **European training domain.** Performance would likely degrade if applied outside the geographic range represented in EuroSAT.

### Disclaimer

EuroSAT is a **benchmark dataset** created for method development and comparison. This project is a **portfolio / learning experiment**, not novel ecological research and not a production vegetation mapping system. The results reported here are consistent with published SmallCNN baselines on EuroSAT RGB and should be interpreted in that context.

## 14  Summary

| Model | Test Accuracy | Macro F1 | Notes |
|---|---|---|---|
| Logistic Regression (mean RGB) | — | — | Spectral mean only |
| Random Forest (mean RGB) | — | — | Spectral mean only |
| SmallCNN (RGB) | — | — | Spatial + spectral |

*(Fill in values after running `python -m src.train && python -m src.evaluate --run_baseline`)*

**Key findings:**

1. A compact CNN trained from scratch outperforms mean-spectral baselines, demonstrating that spatial texture carries discriminative information beyond aggregate reflectance.

2. The largest errors occur among ecologically similar vegetation types (Pasture/HerbaceousVegetation, AnnualCrop/PermanentCrop) — classes that are genuinely ambiguous in RGB imagery without phenological or NIR context.

3. Spectrally distinctive classes (Sea/Lake, Forest, Industrial) are classified with high per-class recall, consistent with the known spectral separability of these cover types.

4. Random image-level splitting likely overestimates real-world generalisation; spatially blocked validation is recommended before applying such a model to landscape-scale mapping.

---
*This notebook is part of the `sentinel-vegetation-cnn` portfolio project.*